# 3.1. Аттрактор Лоренца и горизонт предсказуемости

Исследуем, как малая ошибка в начальном состоянии ограничивает время прогноза для системы Лоренца:
$$
\dot x=\sigma(y-x),\qquad
\dot y=x(\rho-z)-y,\qquad
\dot z=xy-\beta z,
$$
где $\sigma=10$, $\beta=8/3$, $(x(0),y(0),z(0))=(1,1,1)$.

При $\rho=28$ построим фазовую траекторию, проследим рост возмущения начальных данных и оценим старший показатель Ляпунова. Режим при $\rho=10$ используем для сравнения: в нём движение стремится к устойчивому равновесию. Затем проверим, насколько формула горизонта предсказуемости согласуется с прямым расчётом нескольких пар близких траекторий.

Величины безразмерные. Расстояние между состояниями $\mathbf u=(x,y,z)$ измеряем обычной евклидовой нормой, без изменения масштабов координат:
$$d=\|\mathbf v-\mathbf u\|_2
=\sqrt{(v_x-x)^2+(v_y-y)^2+(v_z-z)^2}.$$
Начальное расстояние в опытах без перенормировки обозначим $\delta_0$, а длину возмущения, поддерживаемую при вычислении показателя Ляпунова, — $\varepsilon$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.integrate import solve_ivp
from IPython.display import HTML, Markdown, display

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 10,
    'axes.grid': True, 'grid.alpha': 0.17, 'axes.axisbelow': True,
    'axes.spines.top': False, 'axes.spines.right': False,
    'legend.framealpha': 1, 'legend.handlelength': 3,
    'animation.embed_limit': 60, 'animation.frame_format': 'jpeg',
})
BLUE, ORANGE, GREEN, PURPLE = '#0072B2', '#D55E00', '#008568', '#7B3294'
SIGMA, BETA = 10.0, 8 / 3
STANDARD = dict(rtol=2e-10, atol=2e-12, max_step=0.02)
ACCURATE = dict(rtol=1e-12, atol=1e-14, max_step=0.01)
STRICT = dict(rtol=1e-13, atol=1e-15, max_step=0.005)

def table(headers, rows):
    def line(items):
        return '| ' + ' | '.join(str(x).replace('|', '&#124;') for x in items) + ' |'
    display(Markdown('\n'.join([
        line(headers), line(['---'] * len(headers)), *[line(row) for row in rows]
    ])))

def power10(value):
    return f'$10^{{{int(round(np.log10(value)))}}}$'

## 1. Интегрирование и контроль точности

Используем DOP853 — явный метод Рунге — Кутты восьмого порядка с переменным шагом. Для компоненты состояния $u_i$ масштаб допустимой локальной ошибки задаётся выражением
$$s_i=\mathrm{atol}+\mathrm{rtol}\,|u_i|.$$
Эта оценка служит для выбора шага. Она не ограничивает полную ошибку решения на длинном интервале: в неустойчивом движении малые ошибки предыдущих шагов также усиливаются.

В коде ниже `STANDARD` задаёт обычные настройки, `ACCURATE` — основные настройки расчёта показателя Ляпунова и горизонта, `STRICT` — уточнённые. Начальные точки для сравнений вычисляются с `STRICT` и затем передаются во все варианты в точности одинаковыми.

| Настройки | rtol | atol | Наибольший шаг |
|---|---:|---:|---:|
| Обычные | $2\cdot10^{-10}$ | $2\cdot10^{-12}$ | $0.02$ |
| Основные | $10^{-12}$ | $10^{-14}$ | $0.01$ |
| Уточнённые | $10^{-13}$ | $10^{-15}$ | $0.005$ |

Моменты вывода точек на график не задают шаг интегратора. Значения между его внутренними узлами получаются интерполяцией. В парном расчёте интегрируются шесть уравнений: первые три описывают опорную траекторию, следующие три — возмущённую. Взаимодействия между ними нет.

In [ ]:
def rhs(t, u, rho):
    x, y, z = u
    return np.array([SIGMA * (y - x), x * (rho - z) - y, x * y - BETA * z])


def pair_rhs(t, u, rho):
    return np.concatenate((rhs(t, u[:3], rho), rhs(t, u[3:], rho)))


def integrate(u0, rho, duration, settings=ACCURATE, **kwargs):
    function = rhs if len(u0) == 3 else pair_rhs
    result = solve_ivp(function, (0, duration), u0, args=(rho,),
                       method='DOP853', **settings, **kwargs)
    if not result.success:
        raise RuntimeError(result.message)
    return result

Проверим правую часть на двух простых решениях. При $x=y=0$ остаётся $\dot z=-\beta z$, то есть $z(t)=z(0)e^{-\beta t}$. Кроме того, при $\rho>1$ существуют равновесия
$$C_\pm=\left(\pm\sqrt{\beta(\rho-1)},\;\pm\sqrt{\beta(\rho-1)},\;\rho-1\right),$$
получаемые из условий $y=x$, $z=\rho-1$, $x^2=\beta z$. Подстановка этих точек должна давать нулевую правую часть. Эти проверки позволяют заметить ошибку в знаках или коэффициентах ещё до расчёта хаотической траектории.

In [ ]:
check_t = np.linspace(0, 5, 101)
check = integrate([0, 0, 2], 28, 5, ACCURATE, t_eval=check_t).y.T
exact = np.column_stack((np.zeros_like(check_t), np.zeros_like(check_t),
                         2 * np.exp(-BETA * check_t)))
axis_error = np.linalg.norm(check - exact, axis=1).max()
equilibrium10 = np.array([np.sqrt(BETA * 9), np.sqrt(BETA * 9), 9.])
equilibrium_residual = np.linalg.norm(rhs(0, equilibrium10, 10))
assert axis_error < 1e-10
assert equilibrium_residual < 1e-12
table(['Проверка', 'Максимальное отклонение'], [
    [r'$x=y=0,\ z=2e^{-\beta t}$ на $[0,5]$', f'{axis_error:.2e}'],
    [r'$\lVert f(C_+)\rVert$ при $\rho=10$', f'{equilibrium_residual:.2e}'],
])

## 2. Движение при $\rho=28$

Решим систему на $0\le t\le60$. Начальные обороты около одного равновесия сменяются переходами между двумя областями фазового пространства. Для фазовых портретов отбросим первые 20 единиц времени. Это выбранная длина переходного участка, а не точная граница: у хаотической траектории нет момента «установления», после которого движение становится периодическим.

Три проекции ниже построены по одним и тем же точкам при $20\le t\le60$. Масштабы по обеим осям каждой проекции одинаковы. Под временными рядами приведём диапазоны координат на этом участке: они понадобятся при выборе допустимой ошибки прогноза.

In [ ]:
t_full = np.linspace(0, 60, 12001)
full_solution = integrate([1, 1, 1], 28, 60, STRICT, t_eval=t_full)
trajectory = full_solution.y.T
phase = trajectory[t_full >= 20]

fig, axes = plt.subplots(3, 1, figsize=(10, 6.5), sharex=True, layout='constrained')
for i, (ax, name, color) in enumerate(zip(axes, ['x', 'y', 'z'], [BLUE, ORANGE, GREEN])):
    ax.axvspan(0, 20, color='0.90', zorder=0)
    ax.plot(t_full, trajectory[:, i], color=color, lw=0.9)
    ax.set_ylabel(f'{name}(t)')
axes[-1].set(xlabel='t', xlim=(0, 60))
fig.suptitle(r'$\rho=28$: серым выделен отбрасываемый участок $0\leq t<20$')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.1), layout='constrained')
projections = [(0, 1, 'x', 'y'), (0, 2, 'x', 'z'), (1, 2, 'y', 'z')]
for ax, (i, j, xname, yname) in zip(axes, projections):
    ax.plot(phase[:, i], phase[:, j], color=BLUE, lw=0.55)
    ax.set(xlabel=xname, ylabel=yname, aspect='equal')
fig.suptitle(r'Три проекции одной траектории, $20\leq t\leq60$')
plt.show()

table(['Координата', 'Минимум', 'Максимум', 'Размах'], [
    [name, f'{phase[:, i].min():.2f}', f'{phase[:, i].max():.2f}',
     f'{np.ptp(phase[:, i]):.2f}'] for i, name in enumerate(['x', 'y', 'z'])
])

Чередование лепестков видно по смене знака $x(t)$ и $y(t)$. Интервалы между переключениями неодинаковы; внутри каждого лепестка различаются и амплитуды отдельных оборотов. В проекции $(x,z)$ соседние витки почти накладываются, но замкнутая повторяющаяся орбита на этом интервале не выделяется.

Рисунки показывают нерегулярное ограниченное движение на выбранном промежутке. Чтобы количественно проверить чувствительность к начальным данным, перейдём к расстоянию между двумя решениями.

## 3. Возмущение начальной координаты

Один раз вычислим $\mathbf u_* = \mathbf u(40)$ и запустим из него две траектории:
$$\mathbf u_1(0)=\mathbf u_*,\qquad
\mathbf u_2(0)=\mathbf u_*+(10^{-6},0,0).$$
Далее время отсчитывается от внесения возмущения. В течение 35 единиц времени перенормировка не применяется.

Построим расстояние $d(t)$ при обычных и уточнённых настройках. Отдельно измерим
$$e_{\rm num}(t)=\|\mathbf u_{\rm ordinary}(t)-\mathbf u_{\rm refined}(t)\|,$$
где обе траектории начинаются из одного массива $\mathbf u_*$. Это разность двух численных решений, а не известная ошибка относительно точного решения. Сравнение нужно, чтобы оценить, насколько наблюдаемое расхождение от начального возмущения превосходит влияние настроек интегратора.

In [ ]:
starts = {
    rho: integrate([1, 1, 1], rho, 40, STRICT, t_eval=[40]).y[:, -1]
    for rho in [10., 28.]
}
u_star = starts[28.].copy()
delta0_green = 1e-6
e_x = np.array([1., 0., 0.])
t_pair = np.linspace(0, 35, 7001)
pair_initial = np.r_[u_star, u_star + delta0_green * e_x]
pair_standard = integrate(pair_initial, 28, 35, STANDARD, t_eval=t_pair).y.T
pair_strict = integrate(pair_initial, 28, 35, STRICT, t_eval=t_pair).y.T
d_standard = np.linalg.norm(pair_standard[:, 3:] - pair_standard[:, :3], axis=1)
d_strict = np.linalg.norm(pair_strict[:, 3:] - pair_strict[:, :3], axis=1)
single_standard = integrate(u_star, 28, 35, STANDARD, t_eval=t_pair).y.T
single_strict = integrate(u_star, 28, 35, STRICT, t_eval=t_pair).y.T
e_numerical = np.linalg.norm(single_standard - single_strict, axis=1)

fit_interval = (7., 13.)
fit_mask = (t_pair >= fit_interval[0]) & (t_pair <= fit_interval[1])
fit_rate, fit_intercept = np.polyfit(t_pair[fit_mask], np.log(d_strict[fit_mask]), 1)
log_residual = np.log(d_strict[fit_mask]) - (fit_rate*t_pair[fit_mask] + fit_intercept)
fit_r2 = 1 - np.sum(log_residual**2) / np.sum(
    (np.log(d_strict[fit_mask]) - np.log(d_strict[fit_mask]).mean())**2
)
early = t_pair <= 13
relative_distance_change = np.max(np.abs(d_standard[early] / d_strict[early] - 1))
table(['Величина', 'Результат'], [
    [r'$\mathbf u_*$', '(' + ', '.join(f'{x:.9f}' for x in u_star) + ')'],
    [r'Наклон $\ln d$ на $[7,13]$', f'{fit_rate:.4f}'],
    [r'$R^2$ прямой на $[7,13]$', f'{fit_r2:.3f}'],
    [r'$\max_{0\leq t\leq13}\lvert d_{\rm ordinary}/d_{\rm refined}-1\rvert$',
     f'{relative_distance_change:.2e}'],
    [r'Медиана $d$ на $[20,35]$', f'{np.median(d_strict[t_pair >= 20]):.2f}'],
])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.5), layout='constrained')
for ax, xmax in zip(axes, [35, 14]):
    ax.semilogy(t_pair, d_strict, color=BLUE, lw=2.3, label='Возмущение, уточнённый расчёт')
    ax.semilogy(t_pair, d_standard, color=ORANGE, lw=1.45, ls=(0, (5, 4)),
                zorder=3, label='Возмущение, обычный расчёт')
    nonzero = e_numerical > 0
    ax.semilogy(t_pair[nonzero], e_numerical[nonzero], color=GREEN, lw=1.2, ls='-.',
                label='Разные настройки, одинаковый старт')
    ax.set(xlabel='Время после возмущения', ylabel='Расстояние', xlim=(0, xmax),
           ylim=(1e-13, 1e2))
axes[0].set_title('Рост и насыщение расхождения')
axes[1].set_title('Ранний участок; аппроксимация на [7, 13]')
axes[1].axvspan(*fit_interval, color='0.90', alpha=0.65, zorder=0)
axes[1].semilogy(t_pair[fit_mask], np.exp(fit_intercept + fit_rate*t_pair[fit_mask]),
                 color='black', lw=1.6, ls=(0, (2, 2)), label=fr'Прямая: наклон {fit_rate:.3f}')
axes[0].legend(loc='lower right', fontsize=7.7)
axes[1].legend(loc='lower right', fontsize=7.7)
plt.show()

На участке $7\le t\le13$ расстояние в среднем растёт экспоненциально: линейная аппроксимация $\ln d(t)$ даёт наклон около $0.917$, $R^2\approx0.84$. Это описание выбранного конечного участка. Локальные сжатия и всплески остаются заметными, а наклон зависит от границ аппроксимации; заменять им старший показатель Ляпунова было бы преждевременно.

Начальная стадия устроена сложнее: возмущение сначала меняет направление, а движение по разным участкам лепестков чередует растяжение и сжатие. Поэтому экспоненциальную огибающую нельзя проводить через всю кривую, начиная с $t=0$.

До $t=13$ относительное изменение расстояния при ужесточении настроек не превышает $6\cdot10^{-6}$. При этом $e_{\rm num}(t)$ тоже растёт, но остаётся значительно меньше расстояния в возмущённой паре. Следовательно, ранний рост $d(t)$ воспроизводится при уточнении интегрирования. На больших временах сама траектория уже чувствительна к накопленным численным ошибкам; совпадение только одной длинной реализации не служило бы достаточной проверкой.

После расхождения до размеров лепестков расстояние колеблется на масштабе десятков единиц. Рост ограничен геометрией аттрактора. Две точки могут далеко разойтись по нему и затем снова сблизиться; монотонного роста расстояния не требуется. Порог $d=1$, который далее будет принят для прогноза, задаёт допустимую ошибку и не является универсальной границей применимости линейного приближения.

## 4. Старший показатель Ляпунова

Для малого возмущения $\boldsymbol\eta$ старший показатель характеризует среднюю скорость роста:
$$\lambda_{\max}=\lim_{T\to\infty}\frac1T
\ln\frac{\|\boldsymbol\eta(T)\|}{\|\boldsymbol\eta(0)\|}.$$
Здесь сначала рассматривается линейный предел по длине возмущения. Фиксированную конечную пару нельзя просто интегрировать неограниченно долго и подставить в формулу: после насыщения расстояния такая оценка стремилась бы к нулю.

В методе двух траекторий после каждого интервала $\tau$ измеряем $d_k=\|\mathbf v_k-\mathbf u_k\|$ и возвращаем вспомогательную точку на расстояние $\varepsilon$:
$$\mathbf v_k\leftarrow\mathbf u_k+
\varepsilon\frac{\mathbf v_k-\mathbf u_k}{d_k}.$$
Направление сохраняется, опорная точка не изменяется. За $m$ интервалов произведение коэффициентов растяжения равно $\prod_{k=1}^m(d_k/\varepsilon)$, откуда
$$\widehat\lambda(T)=\frac1T\sum_{k=1}^m\ln\frac{d_k}{\varepsilon},\qquad T=m\tau.$$

Слишком большое $\varepsilon$ выводит возмущение из линейной области, а слишком маленькое ухудшает точность вычитания двух близких состояний. Поэтому длина возмущения также входит в проверку.

Основной расчёт использует $\varepsilon=10^{-7}$, $\tau=0.05$. Начинаем из состояния при исходном $t=40$, затем выполняем ещё 10 единиц времени с перенормировкой, не включая их в сумму. За это время начальное направление $(1,0,0)$ приспосабливается к динамике. Время $T$ на графиках отсчитывается только от начала усреднения. Левый график начинается с $T=10$: до этого накопленная оценка определяется небольшим числом локальных растяжений и испытывает особенно сильные колебания. В сам расчёт среднего входят все интервалы от $T=0$.

In [ ]:
def lyapunov(start, rho, duration, epsilon=1e-7, tau=0.05,
             settings=ACCURATE, alignment_time=10.0):
    n_align = int(round(alignment_time / tau))
    n_average = int(round(duration / tau))
    assert np.isclose(n_align * tau, alignment_time)
    assert np.isclose(n_average * tau, duration)
    u = np.asarray(start).copy()
    v = u + epsilon * np.array([1., 0., 0.])
    growth = np.empty(n_average)
    for k in range(n_align + n_average):
        # Учитываем фактическое расстояние после округления при перенормировке.
        actual_start = np.linalg.norm(v - u)
        result = integrate(np.r_[u, v], rho, tau, settings, t_eval=[tau])
        u, v = result.y[:3, -1], result.y[3:, -1]
        separation = v - u
        distance = np.linalg.norm(separation)
        if distance == 0 or not np.isfinite(distance):
            raise FloatingPointError('Не удалось разрешить расстояние между траекториями')
        if k >= n_align:
            growth[k - n_align] = np.log(distance / actual_start)
        v = u + epsilon * separation / distance
    times = tau * np.arange(1, n_average + 1)
    return dict(t=times, cumulative=np.cumsum(growth) / times, growth=growth, tau=tau)

In [ ]:
long_results = {
    rho: lyapunov(starts[rho], rho, duration=800)
    for rho in [10., 28.]
}

def at_time(result, time):
    index = int(round(time / result['tau'])) - 1
    return result['cumulative'][index]

table(['Время усреднения T', r'$\widehat\lambda$, $\rho=10$',
       r'$\widehat\lambda$, $\rho=28$'], [
    [T, f'{at_time(long_results[10.], T):.6f}', f'{at_time(long_results[28.], T):.6f}']
    for T in [50, 100, 200, 400, 800]
])

fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.4), layout='constrained')
for rho, color in [(10., ORANGE), (28., BLUE)]:
    result = long_results[rho]
    shown = result['t'] >= 10
    axes[0].plot(result['t'][shown], result['cumulative'][shown], color=color, lw=1.25,
                  label=fr'$\rho={rho:g}$')
axes[0].axhline(0, color='0.2', lw=0.8)
axes[0].set(xlabel='Время усреднения T', ylabel=r'$\widehat\lambda(T)$',
             xlim=(10, 800), title='Оба режима, T ≥ 10')
axes[0].legend(loc='lower right')
result = long_results[28.]
tail = result['t'] >= 100
axes[1].plot(result['t'][tail], result['cumulative'][tail], color=BLUE, lw=1.2)
axes[1].set(xlabel='Время усреднения T', ylabel=r'$\widehat\lambda(T)$',
             xlim=(100, 800), title=r'$\rho=28$: увеличение позднего участка')
plt.show()

При $\rho=28$ увеличение $T$ от 200 до 800 меняет оценку с $0.9034$ до $0.9045$, однако промежуточное значение при $T=400$ равно $0.8980$. Сходимость не монотонна: накопленная сумма зависит от доли времени, проведённого на участках с разным растяжением. По этим результатам обоснован вывод $\lambda_{\max}\approx0.90$; последние цифры одной оценки нельзя считать установленной константой.

### Почему при $\rho=10$ показатель отрицателен

Из заданного начального состояния траектория приходит к $C_+=(\sqrt{24},\sqrt{24},9)$. Вблизи него отклонение удовлетворяет линейному приближению $\dot{\boldsymbol\eta}=J(C_+)\boldsymbol\eta$, где
$$J(x,y,z)=
\begin{pmatrix}
-\sigma&\sigma&0\\
\rho-z&-1&-x\\
y&x&-\beta
\end{pmatrix}.$$
Максимальная действительная часть собственного значения этой постоянной матрицы задаёт асимптотическую скорость затухания малого возмущения. Это независимая проверка метода перенормировки в регулярном режиме.

In [ ]:
def jacobian(u, rho):
    x, y, z = u
    return np.array([[-SIGMA, SIGMA, 0], [rho - z, -1, -x], [y, x, -BETA]])

eigenvalues10 = np.linalg.eigvals(jacobian(equilibrium10, 10))
linear_rate10 = eigenvalues10.real.max()
lambda10 = at_time(long_results[10.], 800)
table(['Величина при ρ = 10', 'Значение'], [
    ['Расстояние от u(40) до C₊', f'{np.linalg.norm(starts[10.] - equilibrium10):.2e}'],
    ['Собственные значения J(C₊)', ', '.join(f'{s.real:.6f}{s.imag:+.6f}i' for s in eigenvalues10)],
    ['Максимальная действительная часть', f'{linear_rate10:.6f}'],
    ['Две траектории, T = 800', f'{lambda10:.6f}'],
    ['Модуль разности оценок', f'{abs(lambda10 - linear_rate10):.2e}'],
])

Получено $-0.595244$ методом двух траекторий и $-0.595497$ по матрице Якоби. Разность около $2.5\cdot10^{-4}$ соответствует конечному времени усреднения. Два ведущих собственных значения комплексно сопряжены: направление отклонения вращается, и его норма затухает с колеблющимся множителем. Поэтому оценка по конечному интервалу не обязана точно равняться действительной части собственного значения.

Отрицательный показатель здесь означает притяжение к устойчивой неподвижной точке. Положительная оценка при $\rho=28$ означает среднее усиление малых возмущений вдоль рассмотренной ограниченной траектории и служит численным признаком хаотического режима.

### Влияние длины возмущения, перенормировки и точности

В следующей таблице все варианты начинаются из одних и тех же состояний при $t=40$, используют 10 единиц предварительной ориентации и $T=400$ усреднения. При изменении $\varepsilon$ или $\tau$ остальные параметры сохраняются. Два последних варианта проверяют точность интегрирования: допуски и ограничение шага изменяются совместно.

In [ ]:
variants = [
    ('Основной', 1e-7, 0.05, ACCURATE),
    ('Большее возмущение', 1e-5, 0.05, ACCURATE),
    ('Меньшее возмущение', 1e-9, 0.05, ACCURATE),
    ('Чаще перенормировка', 1e-7, 0.025, ACCURATE),
    ('Реже перенормировка', 1e-7, 0.1, ACCURATE),
    ('Грубее интегрирование', 1e-7, 0.05, dict(rtol=1e-8, atol=1e-10, max_step=0.05)),
    ('Точнее интегрирование', 1e-7, 0.05, STRICT),
]
robustness = []
for index, (name, epsilon, tau, settings) in enumerate(variants):
    values = []
    for rho in [10., 28.]:
        if index == 0:
            value = at_time(long_results[rho], 400)
        else:
            value = lyapunov(starts[rho], rho, 400, epsilon, tau, settings)['cumulative'][-1]
        values.append(value)
    robustness.append(values)
robustness = np.asarray(robustness)
table(['Вариант', 'ε', 'τ', 'rtol', 'atol', 'hₘₐₓ', 'λ̂, ρ = 10', 'λ̂, ρ = 28'], [
    [name, power10(epsilon), f'{tau:g}', power10(settings['rtol']),
     power10(settings['atol']), f"{settings['max_step']:g}", f'{v[0]:.6f}', f'{v[1]:.6f}']
    for (name, epsilon, tau, settings), v in zip(variants, robustness)
])

# Неперекрывающиеся отрезки одной длинной реализации показывают
# собственную изменчивость среднего растяжения на аттракторе.
block_duration = 200
block_rates = long_results[28.]['growth'].reshape(4, -1).sum(axis=1) / block_duration
table(['Интервал усреднения одной реализации', 'Показатель за этот интервал'], [
    [f'[{i*200}, {(i+1)*200}]', f'{value:.6f}'] for i, value in enumerate(block_rates)
])

При $\rho=10$ изменение настроек практически не меняет результат. При $\rho=28$ все варианты дают положительные значения в пределах примерно $0.898$–$0.920$. Это разброс конечновременных оценок, а не доверительный интервал для точного показателя.

Для сравнения, четыре неперекрывающихся отрезка длиной 200 в одной длинной реализации дают $0.893$–$0.922$. Масштаб этой изменчивости сопоставим с разбросом между настройками. На длинном интервале даже очень малая разность численных траекторий меняет последовательность посещаемых участков аттрактора; поэтому сравнение конечных значений не изолирует одну только ошибку интегрирования.

Проверки подтверждают устойчивость знака и величину порядка $0.90$. Уменьшение $\varepsilon$ от $10^{-5}$ до $10^{-9}$ не выявляет систематического смещения, а изменение $\tau$ в четыре раза сохраняет тот же результат с точностью до наблюдаемого разброса. Для дальнейшей оценки времени прогноза используем $\widehat\lambda(800)$ основного расчёта.

## 5. Горизонт предсказуемости

Пусть прогноз считается приемлемым, пока $d(t)<\Delta$. При среднем экспоненциальном росте $d(t)\approx\delta_0e^{\lambda_{\max}t}$ условие $d(T)=\Delta$ даёт
$$T_{\rm pred}\approx\frac{\ln(\Delta/\delta_0)}{\lambda_{\max}}.$$

Выберем $\Delta=1$. На фазовом портрете размахи координат составляют десятки единиц, поэтому этот порог существенно меньше размера аттрактора. Для каждой из двенадцати точек, взятых при исходных временах $40,50,\ldots,150$, зададим возмущение только по $x$, как в разделе 3. Его величины — $10^{-4},10^{-5},10^{-6},10^{-7},10^{-8}$. Всего получаем 60 пар; после запуска их расстояние не перенормируется.

Измеряем **первое** пересечение уровня $d=\Delta$ снизу вверх. Последующие возвраты под порог не отменяют момент первой потери заданной точности. Пересечение определяется по нулю функции $F(t)=d(t)-\Delta$ между внутренними шагами интегратора. Выходная сетка графика на результат не влияет. Возможность пропустить очень короткое пересечение контролируется дополнительным уменьшением шага и допусков.

In [ ]:
def threshold_time(u0, direction, delta, threshold=1., settings=ACCURATE):
    def crossing(t, state, rho):
        return np.linalg.norm(state[3:] - state[:3]) - threshold
    crossing.terminal = True
    crossing.direction = 1
    result = integrate(np.r_[u0, u0 + delta * direction], 28, 40., settings,
                       events=crossing)
    if not len(result.t_events[0]):
        raise RuntimeError('За выбранное время порог не достигнут')
    return result.t_events[0][0]

In [ ]:
start_times = np.arange(40., 151., 10.)
horizon_states = integrate([1, 1, 1], 28, 150, STRICT, t_eval=start_times).y.T
deltas = 10.**(-np.arange(4, 9))
Delta = 1.
hit_times = np.array([
    [threshold_time(state, e_x, delta, Delta, ACCURATE) for delta in deltas]
    for state in horizon_states
])
lambda_used = at_time(long_results[28.], 800)
theory_times = np.log(Delta/deltas) / lambda_used
mean_times = hit_times.mean(axis=0)
std_times = hit_times.std(axis=0, ddof=1)
table(['δ₀', 'Минимум', 'Среднее', 'Максимум', 'Станд. отклонение', 'Формула'], [
    [power10(delta), f'{column.min():.3f}', f'{column.mean():.3f}', f'{column.max():.3f}',
     f'{column.std(ddof=1):.3f}', f'{theory:.3f}']
    for delta, column, theory in zip(deltas, hit_times.T, theory_times)
])

gain_theory = np.log(10)/lambda_used
empirical_slope, empirical_intercept = np.polyfit(np.log(Delta/deltas), mean_times, 1)
gains = np.diff(hit_times, axis=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.8), layout='constrained',
                         gridspec_kw={'width_ratios': [1.3, 1]})
for i, times in enumerate(hit_times):
    axes[0].semilogx(deltas, times, color='0.72', lw=0.75, marker='.', ms=3,
                    label='Отдельные начальные точки' if i == 0 else None)
axes[0].errorbar(deltas, mean_times, yerr=std_times, color=BLUE, lw=2.1,
                 marker='o', ms=5, capsize=3, label='Среднее ± станд. отклонение')
axes[0].plot(deltas, theory_times, color=ORANGE, ls=(0, (5, 3)), lw=1.8,
              zorder=4, label=r'$\ln(\Delta/\delta_0)/\widehat\lambda$')
axes[0].set(xlabel=r'Начальное расстояние $\delta_0$', ylabel='Первое время достижения Δ',
             title=r'$\Delta=1$, двенадцать начальных точек')
axes[0].invert_xaxis()
axes[0].legend(loc='upper left', fontsize=7.5)
positions = np.arange(4)
axes[1].errorbar(positions, gains.mean(axis=0), yerr=gains.std(axis=0, ddof=1),
                 fmt='o', color=BLUE, capsize=5, label='Средняя прибавка ± станд. отклонение')
axes[1].axhline(gain_theory, color=ORANGE, lw=1.8, ls=(0, (5, 3)),
                label=fr'$\ln 10/\widehat\lambda={gain_theory:.3f}$')
axes[1].set(xticks=positions,
             xticklabels=[r'$10^{-4}\to10^{-5}$', r'$10^{-5}\to10^{-6}$',
                          r'$10^{-6}\to10^{-7}$', r'$10^{-7}\to10^{-8}$'],
             xlabel='Уменьшение начальной ошибки', ylabel='Прибавка времени прогноза',
             title='Выигрыш от одного десятичного разряда')
axes[1].tick_params(axis='x', labelsize=8, rotation=15)
axes[1].legend(loc='upper left', fontsize=7.5)
gain_upper = (gains.mean(axis=0) + gains.std(axis=0, ddof=1)).max()
gain_lower = (gains.mean(axis=0) - gains.std(axis=0, ddof=1)).min()
axes[1].set_ylim(min(0, gain_lower), gain_upper * 1.30)
plt.show()

При уменьшении $\delta_0$ от $10^{-4}$ до $10^{-8}$ среднее время первого достижения порога растёт примерно от 10.1 до 20.0. Разброс между точками составляет несколько единиц времени. Это разброс результатов выбранных двенадцати запусков; он не является погрешностью среднего и не задаёт гарантированный горизонт для произвольной начальной точки.

Средняя зависимость близка к прямой по $\ln(\Delta/\delta_0)$. Её наклон даёт эффективную скорость роста около $0.905$, согласующуюся с методом перенормировки. Однако эта формула использует долговременное среднее растяжение, тогда как первое пересечение может произойти во время короткого локального всплеска. Поэтому отдельные серые линии заметно отклоняются от теории и не обязаны иметь одинаковый наклон на каждом промежутке.

Уменьшение начальной ошибки в десять раз добавляет
$$T_{\rm pred}(\delta_0/10)-T_{\rm pred}(\delta_0)
=\frac{\ln10}{\lambda_{\max}}\approx2.55.$$
В прямых расчётах средние прибавки для четырёх последовательных уменьшений ошибки лежат примерно между 2.06 и 3.16. Логарифмическая зависимость описывает общий выигрыш; фиксированная прибавка $2.55$ не гарантируется для каждой пары.

### Проверка точности времени пересечения

Для первой, шестой и двенадцатой начальных точек повторим расчёт при крайних значениях $\delta_0=10^{-4}$ и $10^{-8}$ с настройками `STRICT`. Начальные состояния и направления возмущения остаются теми же. Эта проверка включает как устойчивость парного интегрирования, так и поиск первого пересечения.

In [ ]:
crossing_checks = []
crossing_rows = []
for i in [0, 5, 11]:
    for j in [0, 4]:
        refined = threshold_time(horizon_states[i], e_x, deltas[j], Delta, STRICT)
        original = hit_times[i, j]
        crossing_checks.append(abs(refined - original))
        crossing_rows.append([int(start_times[i]), power10(deltas[j]),
                              f'{original:.6f}', f'{refined:.6f}',
                              f'{abs(refined-original):.2e}'])
table(['Исходное время точки', 'δ₀', 'Основной расчёт', 'Уточнённый', 'Модуль изменения'], crossing_rows)
table(['Дополнительное сравнение', 'Значение'], [
    ['1 / наклон средней зависимости T от ln(Δ/δ₀)', f'{1/empirical_slope:.4f}'],
    ['λ̂(800) из перенормировки', f'{lambda_used:.4f}'],
    ['Прибавка ln(10) / λ̂(800)', f'{gain_theory:.4f}'],
    ['Средняя прибавка по всем последовательным уменьшениям δ₀', f'{gains.mean():.4f}'],
])

Наибольшее изменение времени пересечения в проверенных парах составляет около $2.4\cdot10^{-6}$. Это на много порядков меньше разброса между начальными точками, поэтому наблюдаемую вариацию горизонта нельзя объяснить выбранными допусками интегратора. Проверка относится к шести указанным парам и не означает доказанной границы ошибки для всех возможных начальных данных.

## 6. Анимация расхождения

Показана та же пара, что в разделе 3: начальная точка $\mathbf u(40)$ и сдвиг на $10^{-6}$ по координате $x$. Слева — проекция $(x,z)$ со следом за последние 3 единицы времени, справа — полное расстояние в трёхмерном пространстве состояний. Синяя сплошная линия соответствует опорной траектории, оранжевый пунктир — возмущённой.

Близость следов в одной проекции не всегда означает малое полное расстояние, поскольку координата $y$ на левой панели не показана. График справа позволяет проверить это в тот же момент времени. Вертикальная отметка первого пересечения $d=1$ сохраняется и после возможного возвращения расстояния ниже порога.

In [ ]:
animation_hit = threshold_time(u_star, e_x, delta0_green, Delta, STRICT)
frame_indices = np.linspace(0, len(t_pair) - 1, 241, dtype=int)
trail_points = int(round(3.0 / (t_pair[1] - t_pair[0])))
u_animation, v_animation = pair_strict[:, :3], pair_strict[:, 3:]
all_points = np.vstack((phase, u_animation, v_animation))

fig, (ax_phase, ax_dist) = plt.subplots(1, 2, figsize=(10.8, 4.5))
fig.subplots_adjust(left=0.065, right=0.98, bottom=0.14, top=0.86, wspace=0.26)
ax_phase.plot(phase[:, 0], phase[:, 2], color='0.83', lw=0.4)
ref_line, = ax_phase.plot([], [], color=BLUE, lw=2.5, label='Опорная')
pert_line, = ax_phase.plot([], [], color=ORANGE, lw=1.6, ls=(0, (5, 4)),
                            label='Возмущённая', zorder=4)
ref_point, = ax_phase.plot([], [], 'o', ms=6.5, color=BLUE, mec='white', mew=0.6, zorder=5)
pert_point, = ax_phase.plot([], [], 's', ms=6.5, mfc='none', mec=ORANGE, mew=1.5, zorder=6)
ax_phase.set(xlabel='x', ylabel='z', title='Проекция (x, z)', aspect='equal',
              xlim=(all_points[:, 0].min()-1.5, all_points[:, 0].max()+1.5),
              ylim=(all_points[:, 2].min()-1.5, all_points[:, 2].max()+1.5))
ax_phase.legend(loc='upper right', fontsize=8)
distance_line, = ax_dist.semilogy([], [], color=PURPLE, lw=1.8)
distance_point, = ax_dist.semilogy([], [], 'o', color=PURPLE, ms=5)
cursor = ax_dist.axvline(0, color='0.3', ls=':', lw=1.2)
ax_dist.axhline(Delta, color=ORANGE, lw=1.2, ls=(0, (5, 4)), label='Δ = 1')
ax_dist.axvline(animation_hit, color=GREEN, lw=1.2, ls='-.',
                label=f'Первое пересечение: {animation_hit:.2f}')
ax_dist.set(xlabel='Время после возмущения', ylabel='d(t)',
             xlim=(0, 35), ylim=(d_strict.min()/2, 100), title='Полное расстояние между состояниями')
ax_dist.legend(loc='lower right', fontsize=8)
heading = fig.suptitle('')

def update(frame):
    k = frame_indices[frame]
    first = max(0, k - trail_points)
    ref_line.set_data(u_animation[first:k+1, 0], u_animation[first:k+1, 2])
    pert_line.set_data(v_animation[first:k+1, 0], v_animation[first:k+1, 2])
    ref_point.set_data([u_animation[k, 0]], [u_animation[k, 2]])
    pert_point.set_data([v_animation[k, 0]], [v_animation[k, 2]])
    distance_line.set_data(t_pair[:k+1], d_strict[:k+1])
    distance_point.set_data([t_pair[k]], [d_strict[k]])
    cursor.set_xdata([t_pair[k], t_pair[k]])
    heading.set_text(f't = {t_pair[k]:.2f};   d = {d_strict[k]:.3g}')
    return ref_line, pert_line, ref_point, pert_point, distance_line, distance_point, cursor, heading

movie = FuncAnimation(fig, update, frames=len(frame_indices), interval=70, blit=False)
with plt.rc_context({'savefig.dpi': 90}):
    movie_html = movie.to_jshtml(fps=14, default_mode='once')
plt.close(fig)
display(HTML(movie_html))

## 7. Выводы

При $\rho=28$ в опыте с возмущением $10^{-6}$ по $x$ участок $7\le t\le13$ показывает приблизительно экспоненциальное увеличение расстояния со средним наклоном $\ln d$ около $0.917$. Затем расстояние достигает масштаба аттрактора и колеблется. Уточнение интегрирования сохраняет ранний рост; сравнение решений из одинаковых начальных данных показывает отдельное усиление численной погрешности.

Метод перенормировки даёт $\widehat\lambda(800)\approx-0.5952$ при $\rho=10$ и $\widehat\lambda(800)\approx0.9045$ при $\rho=28$. Для первого режима результат согласуется с линейной скоростью затухания около равновесия. Для второго знак остаётся положительным при всех проверенных длинах возмущения, интервалах перенормировки и настройках интегратора. Разброс около нескольких сотых ограничивает точность, с которой здесь можно сообщать асимптотический показатель.

В 60 опытах без перенормировки зависимость среднего времени достижения $\Delta=1$ согласуется с $\ln(\Delta/\delta_0)/\lambda_{\max}$. Формула предсказывает прибавку $2.55$ единицы времени при десятикратном уточнении начальных данных; средняя прибавка по всем последовательным уменьшениям ошибки в расчёте равна $2.46$. Для отдельной начальной точки выигрыш может существенно отличаться. Увеличение точности начального состояния продлевает прогноз логарифмически.

Условие: [README курса, задание 3.1](https://github.com/yartmihant/Students6Year2026/blob/ff264440f73a8c78f64ac121422a6dc89d2c2c73/README.md). Порядок метода, контроль локальной ошибки и определение событий: [документация SciPy, solve_ivp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html).